# Field Shift — tabla mensual por parcela (Earth Engine + NASA POWER)

Este cuaderno corre los guiones de `/data-pipeline` **en Colab**, no en las laptops de 8 GB.

Antes de empezar necesitas (ver `HUMANO.md`, H-11):
1. Una cuenta de Google con un **proyecto de Earth Engine** registrado para uso no comercial.
2. Acceso de lectura al repositorio del equipo.

Orden: instalar → autenticar → **ver cobertura** → exportar → construir la tabla.

In [ ]:
#@title 1. Parámetros
REPO_URL = "https://github.com/H0M10/MorrisTeamDfs.git"  #@param {type:"string"}
EE_PROJECT = "luca-497507"  #@param {type:"string"}
PARCELS = "parcels/diamond_valley.geojson"  #@param {type:"string"}
UNLABELED = False  #@param {type:"boolean"}
# Déjalos en 0 hasta ver la cobertura del paso 4.
START_YEAR = 0  #@param {type:"integer"}
END_YEAR = 0  #@param {type:"integer"}
assert EE_PROJECT, "Escribe tu proyecto de Earth Engine"

In [ ]:
#@title 2. Clonar el repositorio e instalar
import os
if not os.path.exists("/content/fieldshift"):
    !git clone --depth 1 {REPO_URL} /content/fieldshift
%cd /content/fieldshift/data-pipeline
!pip -q install -r requirements.txt

In [ ]:
#@title 3. Autenticar Earth Engine y montar Drive
import ee
from google.colab import drive
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
drive.mount("/content/drive")
os.environ["EE_PROJECT"] = EE_PROJECT
print("Earth Engine listo:", ee.Number(1).getInfo() == 1)

## 4. Cobertura: qué años cubre cada colección

**No fijes años antes de leer esta salida.** Si alguna colección sale `NO ENCONTRADA`, busca su identificador actual en el catálogo de Earth Engine y corrige `config.py`.

In [ ]:
flag = "--unlabeled" if UNLABELED else ""
!python ee_coverage.py --parcels {PARCELS} {flag} --json coverage.json

## 5. Exportar a Drive

Pon `START_YEAR` y `END_YEAR` en la celda 1 según la cobertura, vuelve a correr esa celda y luego esta. Tarda de minutos a una hora según el número de parcelas.

In [ ]:
assert START_YEAR and END_YEAR, "Fija START_YEAR y END_YEAR en la celda 1 según la cobertura"
flag = "--unlabeled" if UNLABELED else ""
!python ee_export.py --parcels {PARCELS} --start-year {START_YEAR} --end-year {END_YEAR} {flag} --drive-folder fieldshift_exports --wait

In [ ]:
#@title 6. Construir la tabla GeoParquet (junta POWER)
from pathlib import Path
name = Path(PARCELS).stem
!python build_table.py --parcels {PARCELS} --exports /content/drive/MyDrive/fieldshift_exports --out /content/drive/MyDrive/fieldshift_exports/{name}.parquet

In [ ]:
#@title 7. Revisar la tabla
import geopandas as gpd
table = gpd.read_parquet(f"/content/drive/MyDrive/fieldshift_exports/{name}.parquet")
print(table.shape)
table.drop(columns="geometry").head(12)

## Qué sigue

- Descarga el `.parquet` de Drive y pásalo a la persona de Modelos (`/models`).
- Para una parcela piloto que servirá la API, copia su tabla a `backend/data/` (pocos KB).
- **No subas al repositorio** los CSV crudos ni tablas grandes.
- SMAP es de 9 km y POWER de cerca de 50 km: son contexto regional, no de la parcela.